# 11b_gain_tasks — 첫 시도 이득 G를 과제 유형별로 (EV-10)

08이 효용 U로 낸 과제 유형별 표(`report/tables/task_types.csv`, 그림 `task-types.png`)를 **11_gain의 G로** 다시 만든다. 학습도 설정 선택도 없다. 11_gain의 gain lock(보정 점수, τ = .8 규칙, 한 번에 고르기 세 수준)을 그대로 쓴다.

- **칸 나누기:** 원천 × 과제 유형(`categories.py`의 공통 8종). 08의 `by_category`와 같게 칸마다 고정 시작 기준선 H(C)를 다시 만들고, 30문항 미만 칸은 뺀다.
- **원천 합치기:** 원천별 칸 값을 행 수로 가중 평균(08과 같은 방식).
- **구간:** 칸 안에서 질문 묶음 bootstrap(라우터가 같은 표본을 공유), 합친 값은 칸 표본의 가중 평균. 주요 라우터 쌍 차이도 같이 낸다.
- **기술 통계다.** 판정에 쓰지 않는다. test를 다시 읽지만 아무것도 고르지 않고 lock만 적용한다.

**런타임: CPU.** 리허설 1~2분, test 5~10분. 출력 `gain-test-tasks/`: `by-task.parquet`, `by-task-pairs.parquet`, `by-task-summary.md`, `manifest.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
CODE = RUN / 'code'
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
# 기반 소스: 09_seeds 소스 묶음(라우터 최종 실험 코드의 마지막 수정 뒤에 만든 것)
BASE_ZIP = CODE / 'router-final-source-seeds.zip'
assert sha(BASE_ZIP) == BASE_ZIP.with_suffix('.sha256').read_text().split()[0], '기반 소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(BASE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
bundle = json.loads((REPO / 'source-bundle-manifest.json').read_text())
for name, digest in bundle.items():
    assert sha(REPO / name) == digest, name
# gain_eval.py가 그대로 불러 쓰는 모듈은 기반 묶음과 저장소(main)의 판이 같아야 한다
SAME = {
    'lock.py': '04197717a5659f6e294fcd530917d5a49b7c07be40212856e9c8e61f0c234232',
    'test_eval.py': 'c5dc3451abc9bc2058f70f2b3bf0e80a854d4e8dec707622434ea1dba4a8391f',
    'common_track.py': 'fad9e50679a521d950d7162f249e6df2d5afa62925f7ce3d595d875de700360c',
    'categories.py': 'eb2cea4cb798fa84a3c5eec2195ed7ef7a449506390c351bc39c02152fe8a01b',
    'prepare.py': '2dd3fcbfa57f4a7d49a2156ebfa3c23a07438fd19a35eded92a178169e125054',
    'config.json': '43c9087e6bf2e8446b41ed867a0a296ad89f77a0322f61011109d70fe2138562',
}
for name, digest in SAME.items():
    assert bundle[f'ml/notebooks/router_final/{name}'] == digest, f'{name}: 기반 묶음의 판이 다르다'
# 11_gain에서 쓴 gain_eval.py와 이번에 더한 gain_tasks.py
GAIN = CODE / 'gain_eval.py'
assert sha(GAIN) == 'eaada245d737ee9d44588f2381ad2790413688a21d6d49342d43d01220bb5330', 'gain_eval.py가 바뀌었다'
shutil.copy(GAIN, REPO / 'ml/notebooks/router_final/gain_eval.py')
TASKS = CODE / 'gain_tasks.py'
assert sha(TASKS) == 'd0a952de1f6ee667821cb0434b37173e346af99aee9cdda780c1f0a45fd34462', 'gain_tasks.py가 바뀌었다'
shutil.copy(TASKS, REPO / 'ml/notebooks/router_final/gain_tasks.py')
for name in ('gain-lock/gain-lock.json', 'gain-test/manifest.json'):
    assert (RUN / name).exists(), f'11_gain을 먼저 끝내야 한다: {name}'
DATA_ROOT = RUN / 'inputs' / 'test-data'
for name, digest in json.loads((DATA_ROOT / 'test-data-manifest.json').read_text())['files'].items():
    assert sha(DATA_ROOT / name) == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.6.1', 'numpy==2.1.2', 'pandas==2.3.3', 'pyarrow==21.0.0'], check=True)

설치가 끝나면 아래 셀부터 이어서 실행한다. 계산은 새 프로세스(`subprocess`)에서 하므로 설치한 버전이 바로 쓰이고, 세션을 다시 시작하지 않아도 된다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
DATA_ROOT = RUN / 'inputs' / 'test-data'

def run_logged(*args):
    """Run gain_tasks.py and stream its log into this cell (subprocess.run hides it in Colab)."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final/gain_tasks.py'), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

## 1. 리허설 (dev, test를 읽지 않음)

경로·환경 확인용. bootstrap 200회, Colab 디스크에만 쓰고 해석하지 않는다.

In [ ]:
import shutil
REHEARSAL = Path('/content/rehearse-gain-tasks')
shutil.rmtree(REHEARSAL, ignore_errors=True)
run_logged('--run-dir', RUN, '--repo', REPO, '--data-root', DATA_ROOT, '--output-dir', REHEARSAL,
           '--rehearse-on-dev', '--iterations', 200)
print((REHEARSAL / 'by-task-summary.md').read_text()[:1500])

## 2. test (과제 유형별 G)

bootstrap 10,000회. `gain-test-tasks/`가 이미 있으면 멈춘다.

In [ ]:
GAIN_TASKS = RUN / 'gain-test-tasks'
assert not GAIN_TASKS.exists(), '이미 만들었다. 실패한 실행을 다시 할 때만 폴더를 다른 이름으로 옮긴 뒤 실행'
run_logged('--run-dir', RUN, '--repo', REPO, '--data-root', DATA_ROOT, '--output-dir', GAIN_TASKS)
print((GAIN_TASKS / 'by-task-summary.md').read_text())

## 다음

`by-task-summary.md`를 보고한다. 저장소로 `by-task.parquet`·`by-task-pairs.parquet`를 CSV로 옮기고 U 그림과 같은 모양의 G 그림을 만든다.